# PCA of Model Embeddings\n\nUse this notebook to load trained checkpoints, run inference on a dataset split, capture the embedding that feeds each model classifier, and visualize those embeddings in 2D with PCA. Points are colored by the post-disaster damage label.

## 1. Imports

In [ ]:
from pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport yaml\nfrom sklearn.decomposition import PCA\nfrom sklearn.preprocessing import StandardScaler\n\nfrom databases.xBDClimate_database import DAMAGE_CLASSES, get_dataloaders\nfrom models import create_model\n\ntorch.set_grad_enabled(False)\nDEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\nDEVICE

## 2. Configuration\n\nEdit `MODEL_RUNS` with the checkpoints you want to compare. The checkpoint can be either a Lightning `.ckpt` file or a plain PyTorch state dict.

In [ ]:
CONFIG_PATH = Path('configs/config_baseline.yaml')\n\n# Use 'test' for a finite dataloader. Train/val use streaming samplers, so keep MAX_BATCHES set.\nSPLIT = 'test'\n\n# Limit inference for quick exploration. Set both to None to process the whole split.\nMAX_BATCHES = 50\nMAX_SAMPLES = 2000\n\n# Fill in your checkpoint paths here.\nMODEL_RUNS = [\n    {\n        'name': 'image_difference_weather_gru',\n        'arch': 'ImageDifferenceWeatherGRUModel',\n        'checkpoint': 'experiments/xBDClimate/ImageDifferenceWeatherGRUModel/path/to/checkpoints/last.ckpt',\n    },\n    # {\n    #     'name': 'image_difference',\n    #     'arch': 'ImageDifferenceCNNModel',\n    #     'checkpoint': 'experiments/xBDClimate/ImageDifferenceCNNModel/path/to/checkpoints/last.ckpt',\n    # },\n    # {\n    #     'name': 'weather_gru',\n    #     'arch': 'WeatherGRUModel',\n    #     'checkpoint': 'experiments/xBDClimate/WeatherGRUModel/path/to/checkpoints/last.ckpt',\n    # },\n]\n\nwith CONFIG_PATH.open() as f:\n    cfg = yaml.load(f, Loader=yaml.FullLoader)\n\ncfg['data']['augment'] = False\nprint('Device:', DEVICE)\nprint('Data cache:', cfg['data']['cache_dir'])

## 3. Dataloader

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(\n    cache_dir=cfg['data']['cache_dir'],\n    batch_size=cfg['data']['batch_size'],\n    num_workers=cfg['data']['num_workers'],\n    augment=False,\n)\n\nLOADERS = {\n    'train': train_loader,\n    'val': val_loader,\n    'test': test_loader,\n}\n\nloader = LOADERS[SPLIT]\nprint(f'Using split: {SPLIT}')\nprint(f'Batch size: {loader.batch_size if hasattr(loader, "batch_size") else cfg["data"]["batch_size"]}')

## 4. Model Loading and Embedding Capture\n\nThe hook below captures the tensor that goes into the first `Linear` layer of `model.classifier`. That makes the notebook work across the baseline, image-only, weather-only, and combined architectures even though their embedding dimensions differ.

In [ ]:
def build_model(arch_name, cfg, device):\n    model = create_model(\n        arch_name=arch_name,\n        dropout_rate=cfg['arch']['dropout_rate'],\n        backbone=cfg['arch']['backbone'],\n        num_classes=cfg['arch']['num_classes'],\n        vis_dim=cfg['arch'].get('vis_dim', 256),\n        climate_dim=cfg['arch'].get('climate_dim', 128),\n    )\n    return model.to(device)\n\n\ndef strip_lightning_prefix(state_dict):\n    if any(k.startswith('model.') for k in state_dict):\n        return {k[6:] if k.startswith('model.') else k: v for k, v in state_dict.items()}\n    return state_dict\n\n\ndef load_checkpoint(model, checkpoint_path, device):\n    checkpoint_path = Path(checkpoint_path)\n    if not checkpoint_path.exists():\n        raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')\n\n    checkpoint = torch.load(checkpoint_path, map_location=device)\n    state_dict = checkpoint.get('state_dict', checkpoint)\n    state_dict = strip_lightning_prefix(state_dict)\n    load_result = model.load_state_dict(state_dict, strict=False)\n    model.eval()\n    return load_result\n\n\ndef first_classifier_linear(model):\n    if not hasattr(model, 'classifier'):\n        raise AttributeError('Model does not expose a classifier module.')\n    for module in model.classifier.modules():\n        if isinstance(module, nn.Linear):\n            return module\n    raise AttributeError('Could not find a Linear layer inside model.classifier.')\n\n\ndef collect_classifier_embeddings(model, loader, device, max_batches=None, max_samples=None):\n    captured = {}\n\n    def hook(module, inputs, output):\n        captured['embedding'] = inputs[0].detach().cpu()\n\n    handle = first_classifier_linear(model).register_forward_hook(hook)\n    embeddings = []\n    labels = []\n\n    try:\n        for batch_idx, batch in enumerate(loader):\n            if max_batches is not None and batch_idx >= max_batches:\n                break\n\n            patches_pre, patches_post, mask_patches, climate_series, event_labels, labels_pre, labels_post, *_ = batch\n            patches_pre = patches_pre.to(device, non_blocking=True).float()\n            patches_post = patches_post.to(device, non_blocking=True).float()\n            climate_series = climate_series.to(device, non_blocking=True).float()\n            event_labels = event_labels.to(device, non_blocking=True).float()\n\n            _ = model(patches_pre, patches_post, climate_series, event_labels)\n            embeddings.append(captured['embedding'])\n            labels.append(labels_post.detach().cpu())\n\n            if max_samples is not None and sum(x.shape[0] for x in labels) >= max_samples:\n                break\n    finally:\n        handle.remove()\n\n    X = torch.cat(embeddings, dim=0)\n    y = torch.cat(labels, dim=0)\n    if max_samples is not None:\n        X = X[:max_samples]\n        y = y[:max_samples]\n    return X.numpy(), y.numpy()

## 5. Run Inference

In [ ]:
results = {}\n\nfor run in MODEL_RUNS:\n    name = run['name']\n    arch = run['arch']\n    checkpoint = run['checkpoint']\n\n    print(f'Loading {name} ({arch})')\n    model = build_model(arch, cfg, DEVICE)\n    load_result = load_checkpoint(model, checkpoint, DEVICE)\n    print('  missing keys:', len(load_result.missing_keys))\n    print('  unexpected keys:', len(load_result.unexpected_keys))\n\n    X, y = collect_classifier_embeddings(\n        model,\n        loader,\n        DEVICE,\n        max_batches=MAX_BATCHES,\n        max_samples=MAX_SAMPLES,\n    )\n    results[name] = {'arch': arch, 'X': X, 'y': y}\n    print(f'  embeddings: {X.shape}, labels: {y.shape}')\n\nprint('Done. Models loaded:', list(results))

## 6. PCA Projection and Plot

In [ ]:
id_to_damage = {idx: name for name, idx in DAMAGE_CLASSES.items()}\nclass_ids = sorted(id_to_damage)\ncolors = plt.get_cmap('tab10')(np.linspace(0, 1, len(class_ids)))\n\n\ndef pca_2d(X):\n    X_scaled = StandardScaler().fit_transform(X)\n    pca = PCA(n_components=2, random_state=42)\n    Z = pca.fit_transform(X_scaled)\n    return Z, pca\n\n\nif not results:\n    raise ValueError('No results to plot. Add checkpoint paths to MODEL_RUNS and run inference first.')\n\nfig, axes = plt.subplots(1, len(results), figsize=(7 * len(results), 6), squeeze=False)\n\nfor ax, (name, result) in zip(axes.ravel(), results.items()):\n    X = result['X']\n    y = result['y']\n    Z, pca = pca_2d(X)\n\n    for color, class_id in zip(colors, class_ids):\n        mask = y == class_id\n        if mask.any():\n            ax.scatter(\n                Z[mask, 0],\n                Z[mask, 1],\n                s=14,\n                alpha=0.65,\n                color=color,\n                label=id_to_damage[class_id],\n                edgecolors='none',\n            )\n\n    explained = pca.explained_variance_ratio_ * 100\n    ax.set_title(f'{name}\\nPC1 {explained[0]:.1f}% | PC2 {explained[1]:.1f}%')\n    ax.set_xlabel('PC1')\n    ax.set_ylabel('PC2')\n    ax.grid(alpha=0.25)\n    ax.legend(title='Damage', fontsize=9)\n\nplt.tight_layout()\nplt.show()

## 7. Optional: Save PCA Coordinates

In [ ]:
SAVE_COORDINATES = False\nOUTPUT_DIR = Path('pca_embeddings')\n\nif SAVE_COORDINATES:\n    OUTPUT_DIR.mkdir(exist_ok=True)\n    for name, result in results.items():\n        Z, pca = pca_2d(result['X'])\n        y = result['y']\n        output = np.column_stack([Z[:, 0], Z[:, 1], y])\n        out_path = OUTPUT_DIR / f'{name}_{SPLIT}_pca.csv'\n        np.savetxt(out_path, output, delimiter=',', header='pc1,pc2,label', comments='')\n        print(out_path)